In [1]:
!pip install transformers datasets evaluate peft -q

import time
import torch
import numpy as np
import evaluate

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType, PromptTuningConfig

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.3 MB/s eta 0:00:00


In [2]:
dataset = load_dataset("squad")

dataset["train"] = dataset["train"].select(range(2000))
dataset["validation"] = dataset["validation"].select(range(200))

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

In [3]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [4]:
def preprocess_function(examples):
    inputs = tokenizer(
        examples["question"],
        examples["context"],
        max_length=384,
        truncation="only_second",
        padding="max_length",
        return_offsets_mapping=True
    )

    offset_mapping = inputs["offset_mapping"]
    answers = examples["answers"]

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        answer = answers[i]

        if len(answer["answer_start"]) == 0:
            start_positions.append(0)
            end_positions.append(0)
            continue

        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])

        sequence_ids = inputs.sequence_ids(i)

        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx

        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        if offsets[context_start][0] > start_char or offsets[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            idx = context_start
            while idx <= context_end and offsets[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offsets[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions

    inputs.pop("offset_mapping")
    return inputs

In [5]:
tokenized_datasets = dataset.map(preprocess_function, batched=True)

train_data = tokenized_datasets["train"]
eval_data = tokenized_datasets["validation"]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [6]:
class QA_Trainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        start_positions = inputs.pop("start_positions")
        end_positions = inputs.pop("end_positions")

        outputs = model(**inputs)

        loss_fct = torch.nn.CrossEntropyLoss()

        start_loss = loss_fct(outputs.start_logits, start_positions)
        end_loss = loss_fct(outputs.end_logits, end_positions)

        loss = (start_loss + end_loss) / 2

        return (loss, outputs) if return_outputs else loss

In [7]:
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,   # GPU friendly
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    logging_steps=50,
    save_strategy="no",
)

In [8]:
metric = evaluate.load("squad")

def compute_metrics_from_logits(start_logits, end_logits):
    predictions = []
    references = []

    for i in range(len(start_logits)):
        start_indexes = np.argsort(start_logits[i])[-10:]
        end_indexes = np.argsort(end_logits[i])[-10:]

        best_score = -1e9
        best_answer = ""

        input_ids = eval_data[i]["input_ids"]

        for s in start_indexes:
            for e in end_indexes:
                if s <= e and (e - s) < 30:
                    score = start_logits[i][s] + end_logits[i][e]

                    if score > best_score:
                        best_score = score
                        best_answer = tokenizer.decode(
                            input_ids[s:e+1],
                            skip_special_tokens=True
                        )

        predictions.append({
            "id": str(i),
            "prediction_text": best_answer
        })

        real_answer = dataset["validation"][i]["answers"]["text"][0]

        references.append({
            "id": str(i),
            "answers": {
                "text": [real_answer],
                "answer_start": [0]
            }
        })

    return metric.compute(predictions=predictions, references=references)

In [9]:
model = AutoModelForQuestionAnswering.from_pretrained("bert-base-uncased")

trainer = QA_Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=eval_data,
)

start = time.time()
trainer.train()
baseline_time = time.time() - start

pred = trainer.predict(eval_data)
baseline_metrics = compute_metrics_from_logits(*pred.predictions)

print("\n===== BASELINE =====")
print(f"F1 Score    : {baseline_metrics['f1']:.2f}")
print(f"Accuracy(EM): {baseline_metrics['exact_match']:.2f}")
print(f"Training Time: {baseline_time:.2f}")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

Step,Training Loss
50,4.306525
100,3.374685
150,2.509656
200,2.021313
250,1.796422
300,1.193585
350,1.197754
400,1.105073
450,1.088671
500,0.979294



===== BASELINE =====
F1 Score    : 66.83
Accuracy(EM): 56.50
Training Time: 293.64


In [10]:
model_lora = AutoModelForQuestionAnswering.from_pretrained("bert-base-uncased")

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type="QUESTION_ANS"
)

model_lora = get_peft_model(model_lora, lora_config)

trainer_lora = QA_Trainer(
    model=model_lora,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=eval_data,
)

start = time.time()
trainer_lora.train()
lora_time = time.time() - start

pred = trainer_lora.predict(eval_data)
lora_metrics = compute_metrics_from_logits(*pred.predictions)

print("\n===== LoRA =====")
print(f"F1 Score    : {lora_metrics['f1']:.2f}")
print(f"Accuracy(EM): {lora_metrics['exact_match']:.2f}")
print(f"Training Time: {lora_time:.2f}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

Step,Training Loss
50,5.819663
100,5.635993
150,5.305778
200,5.011736
250,4.774081
300,4.590026
350,4.528098
400,4.470499
450,4.306682
500,4.253381



===== LoRA =====
F1 Score    : 10.37
Accuracy(EM): 7.00
Training Time: 243.03


In [11]:
model_adapter = AutoModelForQuestionAnswering.from_pretrained("bert-base-uncased")

adapter_config = PromptTuningConfig(
    task_type=TaskType.QUESTION_ANS,
    num_virtual_tokens=20
)

model_adapter = get_peft_model(model_adapter, adapter_config)

trainer_adapter = QA_Trainer(
    model=model_adapter,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=eval_data,
)

start = time.time()
trainer_adapter.train()
adapter_time = time.time() - start

pred = trainer_adapter.predict(eval_data)
adapter_metrics = compute_metrics_from_logits(*pred.predictions)

print("\n===== ADAPTER =====")
print(f"F1 Score    : {adapter_metrics['f1']:.2f}")
print(f"Accuracy(EM): {adapter_metrics['exact_match']:.2f}")
print(f"Training Time: {adapter_time:.2f}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

Step,Training Loss
50,5.925051
100,5.873999
150,5.853069
200,5.799879
250,5.770744
300,5.763672
350,5.739739
400,5.708474
450,5.693160
500,5.688239



===== ADAPTER =====
F1 Score    : 4.19
Accuracy(EM): 0.00
Training Time: 240.98


In [12]:
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

print("\n===== PARAMETERS =====")

print("Baseline:", count_parameters(model))
print("LoRA    :", count_parameters(model_lora))
print("Adapter :", count_parameters(model_adapter))


===== PARAMETERS =====
Baseline: (108893186, 108893186)
LoRA    : (109189636, 296450)
Adapter : (108910084, 16898)


In [13]:
import time
import numpy as np
import evaluate
from transformers import AutoModelForQuestionAnswering
from peft import LoraConfig, get_peft_model, TaskType, PromptTuningConfig

metric = evaluate.load("squad")

# =========================
# METRIC FUNCTION
# =========================
def compute_metrics_from_logits(trainer):
    pred = trainer.predict(eval_data)
    start_logits, end_logits = pred.predictions

    predictions = []
    references = []

    for i in range(len(start_logits)):
        start_indexes = np.argsort(start_logits[i])[-10:]
        end_indexes = np.argsort(end_logits[i])[-10:]

        best_score = -1e9
        best_answer = ""

        input_ids = eval_data[i]["input_ids"]

        for s in start_indexes:
            for e in end_indexes:
                if s <= e and (e - s) < 30:
                    score = start_logits[i][s] + end_logits[i][e]

                    if score > best_score:
                        best_score = score
                        best_answer = tokenizer.decode(
                            input_ids[s:e+1],
                            skip_special_tokens=True
                        )

        predictions.append({
            "id": str(i),
            "prediction_text": best_answer
        })

        real_answer = dataset["validation"][i]["answers"]["text"][0]

        references.append({
            "id": str(i),
            "answers": {
                "text": [real_answer],
                "answer_start": [0]
            }
        })

    results = metric.compute(predictions=predictions, references=references)

    return results["f1"], results["exact_match"]


# =========================
# PARAMETER FUNCTION
# =========================
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


# =========================
# TRAIN + EVALUATE FUNCTION
# =========================
def run_model(model, name):
    trainer = QA_Trainer(
        model=model,
        args=training_args,
        train_dataset=train_data,
        eval_dataset=eval_data,
    )

    start = time.time()
    trainer.train()
    training_time = time.time() - start

    f1, acc = compute_metrics_from_logits(trainer)

    total_params, trainable_params = count_parameters(model)

    return {
        "Model": name,
        "F1": round(f1, 2),
        "Accuracy": round(acc, 2),
        "Time": round(training_time, 2),
        "Total Params": total_params,
        "Trainable Params": trainable_params
    }


# =========================
# BASELINE
# =========================
baseline_model = AutoModelForQuestionAnswering.from_pretrained("bert-base-uncased")
baseline_results = run_model(baseline_model, "Baseline")


# =========================
# LoRA
# =========================
lora_model = AutoModelForQuestionAnswering.from_pretrained("bert-base-uncased")

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type="QUESTION_ANS"
)

lora_model = get_peft_model(lora_model, lora_config)
lora_results = run_model(lora_model, "LoRA")


# =========================
# Adapter (Prompt Tuning)
# =========================
adapter_model = AutoModelForQuestionAnswering.from_pretrained("bert-base-uncased")

adapter_config = PromptTuningConfig(
    task_type=TaskType.QUESTION_ANS,
    num_virtual_tokens=20
)

adapter_model = get_peft_model(adapter_model, adapter_config)
adapter_results = run_model(adapter_model, "Adapter")


# =========================
# FINAL TABLE OUTPUT
# =========================
print("\n===== FINAL RESULTS TABLE =====\n")

for res in [baseline_results, lora_results, adapter_results]:
    print(res)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

Step,Training Loss
50,4.253118
100,3.244269
150,2.627216
200,2.073285
250,1.836568
300,1.239196
350,1.178593
400,1.132077
450,1.076145
500,1.025322


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

Step,Training Loss
50,5.819663
100,5.635993
150,5.305778
200,5.011736
250,4.774081
300,4.590026
350,4.528098
400,4.470499
450,4.306682
500,4.253381


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

Step,Training Loss
50,5.925051
100,5.873999
150,5.853069
200,5.799879
250,5.770744
300,5.763671
350,5.739738
400,5.708474
450,5.693160
500,5.688239



===== FINAL RESULTS TABLE =====

{'Model': 'Baseline', 'F1': 63.14, 'Accuracy': 51.5, 'Time': 298.9, 'Total Params': 108893186, 'Trainable Params': 108893186}
{'Model': 'LoRA', 'F1': 10.37, 'Accuracy': 7.0, 'Time': 244.95, 'Total Params': 109189636, 'Trainable Params': 296450}
{'Model': 'Adapter', 'F1': 4.19, 'Accuracy': 0.0, 'Time': 242.83, 'Total Params': 108910084, 'Trainable Params': 16898}
